In [119]:
# importing required libraries
import pandas as pd
import numpy as np

In [120]:
# attempt to read csv (encoding issue)
#df = pd.read_csv('/Users/nick/Dev/College/Year 2/Semester 4/CC5061 Applied Data Science/Coursework/Milestone 1/DiwaliSalesData.csv')

In [121]:
# import chardet for encoding detection
import chardet as cd

# read raw bytes from csv file
init_data = open('/Users/nick/Dev/College/Year 2/Semester 4/CC5061 Applied Data Science/Coursework/Milestone 1/DiwaliSalesData.csv', 'rb').read()

# detect the most likely encoding from the sample
result = cd.detect(init_data)

# load csv into DataFrame using detected encoding
df = pd.read_csv('/Users/nick/Dev/College/Year 2/Semester 4/CC5061 Applied Data Science/Coursework/Milestone 1/DiwaliSalesData.csv', encoding=result['encoding'])

In [122]:
# display first 15 rows of the DataFrame
df.head(15)

,User_ID,Cust_name,Product_ID,Gender,Age Group,Age,Marital_Status,State,Zone,Occupation,Product_Category,Orders,Amount,Status,unnamed1
0,1002903,Sanskriti,P00125942,F,26-35,28,0,Maharashtra,Western,Healthcare,Auto,1,23952.00,NaN,NaN
1,1000732,Kartik,P00110942,F,26-35,35,1,Andhra Pradesh,Southern,Govt,Auto,3,23934.00,NaN,NaN
2,1001990,Bindu,P00118542,F,26-35,35,1,Uttar Pradesh,Central,Automobile,Auto,3,23924.00,NaN,NaN
3,1001425,Sudevi,P00237842,M,0-17,16,0,Karnataka,Southern,Construction,Auto,2,23912.00,NaN,NaN
4,1000588,Joni,P00057942,M,26-35,28,1,Gujarat,Western,Food Processing,Auto,2,23877.00,NaN,NaN
5,1000588,Joni,P00057942,M,26-35,28,1,Himachal Pradesh,Northern,Food Processing,Auto,1,23877.00,NaN,NaN
6,1001132,Balk,P00018042,F,18-25,25,1,Uttar Pradesh,Central,Lawyer,Auto,4,23841.00,NaN,NaN
7,1002092,Shivangi,P00273442,F,55+,61,0,Maharashtra,Western,IT Sector,Auto,1,NaN,NaN,NaN
8,1003224,Kushal,P00205642,M,26-35,35,0,Uttar Pradesh,Central,Govt,Auto,2,23809.00,NaN,NaN
9,1003650,Ginny,P00031142,F,26-35,26,1,Andhra Pradesh,Southern,Media,Auto,4,23799.99,NaN,NaN


In [123]:
# show dataframe info and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11251 entries, 0 to 11250
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   User_ID           11251 non-null  int64  
 1   Cust_name         11251 non-null  object 
 2   Product_ID        11251 non-null  object 
 3   Gender            11251 non-null  object 
 4   Age Group         11251 non-null  object 
 5   Age               11251 non-null  int64  
 6   Marital_Status    11251 non-null  int64  
 7   State             11251 non-null  object 
 8   Zone              11251 non-null  object 
 9   Occupation        11251 non-null  object 
 10  Product_Category  11251 non-null  object 
 11  Orders            11251 non-null  int64  
 12  Amount            11239 non-null  float64
 13  Status            0 non-null      float64
 14  unnamed1          0 non-null      float64
dtypes: float64(3), int64(4), object(8)
memory usage: 1.3+ MB


In [124]:
# convert Age column to numeric
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# define mapping function for age groups
def age_to_group(a):
    if pd.isna(a):
        return 'Unknown'
    try:
        a = int(a)
    except:
        return 'Unknown'
    if a <= 17:
        return 'Minor'
    if a <= 54:
        return 'Adult'
    return 'Senior'

# apply mapping
df['AgeGroup'] = df['Age'].apply(age_to_group)
# make AgeGroup an ordered categorical (Minor < Adult < Senior)
categories = ['Minor', 'Adult', 'Senior']
df['AgeGroup'] = pd.Categorical(df['AgeGroup'], categories=categories, ordered=True)
# show counts
print(df['AgeGroup'].value_counts(dropna=False))

df

AgeGroup
Adult     10370
Senior      585
Minor       296
Name: count, dtype: int64


,User_ID,Cust_name,Product_ID,Gender,Age Group,Age,Marital_Status,State,Zone,Occupation,Product_Category,Orders,Amount,Status,unnamed1,AgeGroup
0,1002903,Sanskriti,P00125942,F,26-35,28,0,Maharashtra,Western,Healthcare,Auto,1,23952.0,NaN,NaN,Adult
1,1000732,Kartik,P00110942,F,26-35,35,1,Andhra Pradesh,Southern,Govt,Auto,3,23934.0,NaN,NaN,Adult
2,1001990,Bindu,P00118542,F,26-35,35,1,Uttar Pradesh,Central,Automobile,Auto,3,23924.0,NaN,NaN,Adult
3,1001425,Sudevi,P00237842,M,0-17,16,0,Karnataka,Southern,Construction,Auto,2,23912.0,NaN,NaN,Minor
4,1000588,Joni,P00057942,M,26-35,28,1,Gujarat,Western,Food Processing,Auto,2,23877.0,NaN,NaN,Adult
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11246,1000695,Manning,P00296942,M,18-25,19,1,Maharashtra,Western,Chemical,Office,4,370.0,NaN,NaN,Adult
11247,1004089,Reichenbach,P00171342,M,26-35,33,0,Haryana,Northern,Healthcare,Veterinary,3,367.0,NaN,NaN,Adult
11248,1001209,Oshin,P00201342,F,36-45,40,0,Madhya Pradesh,Central,Textile,Office,4,213.0,NaN,NaN,Adult
11249,1004023,Noonan,P00059442,M,36-45,37,0,Karnataka,Southern,Agriculture,Office,3,206.0,NaN,NaN,Adult


In [125]:
# convert Amount to numeric
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

# define simple group edges and labels
groups = [-1, 4000, 10000, float('inf')]
group_labels = ['Low', 'Medium', 'High']

# create Purchase_Value_Category using pd.cut
df['Purchase_Value_Category'] = pd.cut(df['Amount'], bins=groups, labels=group_labels, right=True)

# show counts (NaN means missing/invalid Amount)
print(df['Purchase_Value_Category'].value_counts(dropna=False))



Purchase_Value_Category
Medium    5347
High      4154
Low       1738
NaN         12
Name: count, dtype: int64


In [126]:
# drop irrelevant columns (privacy and artefact)
to_drop = ['Cust_name', 'User_ID', 'unnamed1']
# drop only if present to avoid errors
df.drop(columns=to_drop, inplace=True, errors='ignore')
# show columns and shape after drop
print('columns after drop:', list(df.columns))
print('shape after drop:', df.shape)

columns after drop: ['Product_ID', 'Gender', 'Age Group', 'Age', 'Marital_Status', 'State', 'Zone', 'Occupation', 'Product_Category', 'Orders', 'Amount', 'Status', 'AgeGroup', 'Purchase_Value_Category']
shape after drop: (11251, 14)


In [127]:
# Drop rows missing `Amount` or `Age` (Status column is fully missing in this dataset)
crit = ['Amount','Age']
print('rows before:', len(df))
# drop rows missing Amount or Age
df.dropna(subset=crit, inplace=True)
print('rows after:', len(df))

rows before: 11251
rows after: 11239


In [10]:
# 3.5 List unique values and counts for Product_Category and Zone
pc = 'Product_Category'
zone = 'Zone'

# Verify columns exist in the cleaned DataFrame before summarizing counts
if pc in df.columns and zone in df.columns:
    print('Product_Category counts:\n')
    print(df[pc].value_counts(dropna=False))
    print('\nZone counts:\n')
    print(df[zone].value_counts(dropna=False))
    print('\nProduct_Category unique:', df[pc].dropna().unique())
print('Zone unique:', df[zone].dropna().unique())

Product_Category counts:

Product_Category
Clothing & Apparel       2655
Food                     2493
Electronics & Gadgets    2087
Footwear & Shoes         1064
Household items           520
Beauty                    422
Games & Toys              386
Sports Products           356
Furniture                 353
Pet Care                  212
Office                    113
Stationery                112
Books                     103
Auto                      100
Decor                      96
Veterinary                 81
Tupperware                 72
Hand & Power Tools         26
Name: count, dtype: int64

Zone counts:

Zone
Central     4296
Southern    2695
Western     1955
Northern    1491
Eastern      814
Name: count, dtype: int64

Product_Category unique: ['Auto' 'Hand & Power Tools' 'Stationery' 'Tupperware' 'Footwear & Shoes'
 'Furniture' 'Food' 'Games & Toys' 'Sports Products' 'Books'
 'Electronics & Gadgets' 'Decor' 'Clothing & Apparel' 'Beauty'
 'Household items' 'Pet Care' 'Veter